In [16]:
import pandas as pd
import re

# Load the jobstreet data
file_path = 'jobstreet_2022-08-31.csv'
df = pd.read_csv(file_path)

# Function to extract skills from job description and job title
# This is a simple approach, you can expand the skill list as needed
skill_keywords = [
    'python', 'excel', 'sql', 'marketing', 'sales', 'accounting', 'finance', 'audit', 'tax', 'restaurant', 'hotel',
    'publicity', 'event', 'quality control', 'housekeeping', 'trainee', 'admin', 'hr', 'client', 'manager', 'developer',
    'customer service', 'security', 'design', 'graphic', 'content', 'writer', 'teacher', 'kasir', 'driver', 'logistics',
    'procurement', 'social media', 'business', 'intern', 'assistant', 'staff', 'supervisor', 'operator', 'purchasing',
    'sales executive', 'account manager', 'marketing manager', 'product development', 'accounting staff', 'frontend',
    'backend', 'system', 'support', 'tech', 'it', 'recruiter', 'trainer', 'trainer', 'audit', 'finance', 'admin', 'developer'
]

# Combine jobTitle and description columns for skill extraction
combined_text = df['jobTitle'].fillna('') + ' ' + df['description'].fillna('')

# Extract skills
def extract_skills(text):
    found_skills = set()
    for skill in skill_keywords:
        if re.search(r'\b' + re.escape(skill) + r'\b', text, re.IGNORECASE):
            found_skills.add(skill)
    return ', '.join(sorted(found_skills))

df['skills'] = combined_text.apply(extract_skills)

# Keep only job id, job title, company name, location, and skills
cleaned_df = df[['id', 'jobTitle', 'companyName', 'locations', 'skills']]

# Save cleaned data
cleaned_df.to_csv('jobstreet_2022-08-31_skills.csv', index=False)

print(df['skills'])

# Count how many jobs have no skills extracted
no_skills_count = (df['skills'] == '').sum()
print(f"\nNumber of jobs with no skills found: {no_skills_count}")

0       quality control, staff
1                        audit
2             event, publicity
3                        sales
4        housekeeping, trainee
                 ...          
4820                     admin
4821                          
4822                   manager
4823            manager, sales
4824    sales, sales executive
Name: skills, Length: 4825, dtype: object

Number of jobs with no skills found: 2087


In [17]:
# You should only need to run these install/download commands once.
# import sys
# import subprocess
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'spacy'])
# subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])

import pandas as pd
import spacy
import re
from spacy.lang.en.stop_words import STOP_WORDS

# Load spaCy English model
# Disabling components we don't need can speed up processing
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

# Load the jobstreet data
file_path = 'jobstreet_2022-08-31.csv'
df = pd.read_csv(file_path)

# Combine jobTitle and description columns for skill extraction
df['combined_text'] = df['jobTitle'].fillna('') + ' ' + df['description'].fillna('')

# Add custom stopwords that are common in job descriptions but are not skills
custom_stopwords = set(['job', 'experience', 'work', 'company', 'team', 'candidate', 'skill', 'requirement', 'qualification', 'responsibility', 'description', 'title', 'location', 'salary', 'application', 'education', 'degree', 'year', 'month', 'day', 'etc'])
all_stopwords = STOP_WORDS.union(custom_stopwords)

def extract_skills_spacy(texts):
    skills_list = []
    # Process texts in batches for efficiency
    for doc in nlp.pipe(texts, batch_size=50):
        # Extract noun chunks, lemmatize, and filter
        skills = set()
        for chunk in doc.noun_chunks:
            # Clean the chunk text
            cleaned_chunk = chunk.lemma_.lower().strip()
            # Check if it's a meaningful skill
            if cleaned_chunk and cleaned_chunk not in all_stopwords and len(cleaned_chunk) > 2:
                skills.add(cleaned_chunk)
        skills_list.append(', '.join(sorted(skills)))
    return skills_list

# Apply the optimized skill extraction function
df['skills'] = extract_skills_spacy(df['combined_text'])

# Keep only the essential columns
cleaned_df = df[['id', 'jobTitle', 'companyName', 'locations', 'skills']]

# Save the new cleaned data
cleaned_df.to_csv('jobstreet_2022-08-31_skills_nlp_v2.csv', index=False)

print("NLP skill extraction complete. Cleaned data saved to 'jobstreet_2022-08-31_skills_nlp_v2.csv'")
print(cleaned_df.head())

# Count how many jobs still have no skills found
no_skills_count = (cleaned_df['skills'] == '').sum()
print(f"\nNumber of jobs with no skills found: {no_skills_count}")
print(f"Total jobs processed: {len(df)}")

ValueError: [E029] `noun_chunks` requires the dependency parse, which requires a statistical model to be installed and loaded. For more info, see the documentation:
https://spacy.io/usage/models

In [ ]:
# count null in skills or ''
no_skills_count = (df['skills'] == '').sum()
print(f"\nNumber of jobs with no skills found: {no_skills_count}")


Number of jobs with no skills found: 3662


In [18]:
# 1. Install Sastrawi for Indonesian language processing
import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'sastrawi'])

import pandas as pd
import re
from collections import Counter
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# 2. Setup Indonesian Stemmer and Stopword Remover
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

# Load the data
file_path = 'jobstreet_2022-08-31.csv'
df = pd.read_csv(file_path)

# Combine text fields
df['combined_text'] = df['jobTitle'].fillna('') + ' ' + df['description'].fillna('')

# 3. Discover skill keywords from the data itself
# Process all text to find common terms
all_text = ' '.join(df['combined_text']).lower()
# Remove stopwords first
text_no_stopwords = stopword_remover.remove(all_text)
# Stem the remaining text to get root words
stemmed_text = stemmer.stem(text_no_stopwords)
# Count the frequency of the stemmed words
all_words = re.findall(r'\b[a-z]{3,15}\b', stemmed_text)
word_counts = Counter(all_words)

# Create a list of top 100 most common words as our skill keywords
# You can adjust this number
skill_keywords = [word for word, count in word_counts.most_common(100)]

print("Discovered Skill Keywords (Top 20):")
print(skill_keywords[:20])

# 4. Function to extract skills based on discovered keywords
def extract_skills_sastrawi(text):
    # Prepare the text: lowercase, remove stopwords, and stem
    cleaned_text = stopword_remover.remove(text.lower())
    stemmed_text_for_row = stemmer.stem(cleaned_text)
    
    found_skills = set()
    for skill in skill_keywords:
        if skill in stemmed_text_for_row:
            found_skills.add(skill)
    return ', '.join(sorted(found_skills))

# Apply the function to each row
df['skills'] = df['combined_text'].apply(extract_skills_sastrawi)

# 5. Save the final, cleaned data
cleaned_df = df[['id', 'jobTitle', 'companyName', 'locations', 'skills']]
output_filename = 'jobstreet_2022-08-31_skills_sastrawi.csv'
cleaned_df.to_csv(output_filename, index=False)

print(f"\nNLP skill extraction complete. Cleaned data saved to '{output_filename}'")
print("\nFirst 10 rows of the cleaned data:")
print(cleaned_df.head(10))

# Check effectiveness
no_skills_count = (cleaned_df['skills'] == '').sum()
print(f"\nNumber of jobs with no skills found: {no_skills_count}")
print(f"Total jobs processed: {len(df)}")

Discovered Skill Keywords (Top 20):
['manager', 'sales', 'staff', 'marketing', 'jakarta', 'supervisor', 'engineer', 'executive', 'officer', 'senior', 'admin', 'specialist', 'accounting', 'business', 'area', 'assistant', 'account', 'guru', 'development', 'project']

NLP skill extraction complete. Cleaned data saved to 'jobstreet_2022-08-31_skills_sastrawi.csv'

First 10 rows of the cleaned data:
           id                                        jobTitle  \
0  1032680084                           Quality Control Staff   
1  1032858553                                  Internal Audit   
2  1032856849                     Event & Publicity Executive   
3  1032857282       Sales Reguler Bank Permata (Kab Semarang)   
4  1032856976                       Housekeeping Trainee (HK)   
5  1032857331             Sales Reguler Bank Permata (Kendal)   
6  1032858737                                             CDP   
7  1032856940              HR Generalist (Tulodong - Jakarta)   
8  1032733950  De

In [19]:
df['skills']

0         control, quality, staff
1         audit, intern, internal
2                       executive
3                 kab, les, sales
4                                
                  ...            
4820                        admin
4821           intern, internship
4822    and, development, manager
4823          les, manager, sales
4824        executive, les, sales
Name: skills, Length: 4825, dtype: object

In [21]:
# Create a new dataframe with only the specified columns.
# Note: Assuming 'categoriesName' is a column in the dataframe and 'companiesName' is a typo for 'companyName'.
df = pd.read_csv('mergeFile.csv')

try:
    categories_df = df[['id', 'companyName', 'categoriesName']]
    
    # Save the new dataframe to a CSV file
    output_filename = 'merge.csv'
    categories_df.to_csv(output_filename, index=False)

    print(f"CSV file '{output_filename}' created successfully.")
    print(categories_df.head())
except KeyError as e:
    print(f"An error occurred: {e}. Please ensure the column names exist in the DataFrame.")
    print("Available columns are:", df.columns.tolist())

C:\Users\Aidam\AppData\Local\Temp\ipykernel_45048\4202411128.py:3: DtypeWarning: Columns (5,7,8,9,11,17,19,20,21,22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('mergeFile.csv')


CSV file 'merge.csv' created successfully.
        id             companyName  \
0  3718618        MODENA INDONESIA   
1  3718537  Yayasan Bina Nusantara   
2  3718508  Yayasan Bina Nusantara   
3  3715202              GriyaBayar   
4  3718527  Yayasan Bina Nusantara   

                                      categoriesName  
0  Sumber Daya Manusia/Personalia,Sumber Daya Man...  
1    Komputer/Teknologi Informasi,IT-Perangkat Lunak  
2                    Pendidikan/Pelatihan,Pendidikan  
3  Komputer/Teknologi Informasi,IT-Admin Jaringan...  
4                    Pendidikan/Pelatihan,Pendidikan  
